<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="/ipynb/Deep-Learning/02-tensors-computation-graphs-pytorch.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **Tensor、计算图与 PyTorch** {#tensors-computation-graphs-pytorch}

一个深度学习模型可能包含数百万个参数与数百次运算，但它的执行过程建立在一组很小的基本词汇上：Tensor 保存数值状态，Tensor 运算转换这些状态，计算图记录输出如何依赖前面的数值。因此，大多数实现错误都可以追溯到以下三个问题之一：

- 每个 Tensor 轴分别表示什么？
- 当前运算是否组合了预期的元素？
- PyTorch 是否记录了梯度计算、序列化和设备迁移所需要的依赖关系？

本章将建立一种在模型变得庞大之前回答这些问题的方法。目标并不是记住所有 PyTorch 方法，而是学会按照**语义 -> shape -> 运算 -> 计算图 -> 系统状态**的顺序推理。后续章节会在这个基础上继续加入神经网络组件、反向传播理论、优化方法和可扩展执行。

### **标量、向量、矩阵与 Tensor** {#scalars-vectors-matrices-tensors}

**Tensor** 是具有统一数据类型的规则多维数组。在深度学习软件中，这个术语也包含我们熟悉的低阶对象：

| 对象 | 阶数 | Shape 示例 | 典型含义 |
|---|---:|---|---|
| 标量 | 0 | `[]` | 一个损失、概率或学习率 |
| 向量 | 1 | `[D]` | 一个特征向量或类别 logit 向量 |
| 矩阵 | 2 | `[B, D]` | 一批特征向量或一个权重矩阵 |
| 三阶 Tensor | 3 | `[B, L, D]` | 一批 token 表示序列 |
| 四阶 Tensor | 4 | `[B, C, H, W]` | 一批 channel-first 图像 |

对于 Tensor $X \in \mathbb{R}^{d_1 \times d_2 \times \cdots \times d_k}$：

- $k$ 是它的**阶数**，也就是轴的数量；
- $(d_1,d_2,\ldots,d_k)$ 是它的 **shape**；
- $d_j$ 是第 $j$ 个轴的长度；
- $\prod_{j=1}^{k} d_j$ 是元素总数。

这里的阶数并不是矩阵的秩。Tensor 的阶数统计轴的数量；矩阵的秩衡量线性无关性。Shape 为 `[2, 3]` 的 Tensor 始终是二阶 Tensor，但根据具体数值，其矩阵秩可能是 1 或 2。

只有数值还不足以完整描述一个 Tensor。在实际系统中，Tensor 还包含：

- **dtype**，例如 `float32`、`bfloat16` 或 `int64`；
- **device**，例如 CPU、CUDA GPU 或 Apple MPS；
- **layout 与 stride**，决定逻辑索引如何映射到存储空间；
- 可选的 autograd 元数据，包括是否需要追踪相关运算。

因此，两个数值完全相同的 Tensor 也可能表现不同：一个可能是位于 CPU 上的整数标签，另一个可能是位于 GPU 上、启用了梯度追踪的浮点激活值。

<details>
<summary><strong>PyTorch：构造 Tensor 并检查其元数据</strong></summary>

~~~python
import torch

scalar = torch.tensor(2.5)                         # shape []
vector = torch.tensor([1.0, -1.0, 0.5])           # shape [3]
matrix = torch.arange(12, dtype=torch.float32).reshape(3, 4)
image_batch = torch.zeros(8, 3, 32, 32)            # [B, C, H, W]
token_batch = torch.zeros(8, 128, 768)             # [B, L, D]


def describe(name: str, tensor: torch.Tensor) -> None:
    """Print the metadata needed for basic tensor debugging."""
    print(
        f"{name:12s}",
        f"shape={tuple(tensor.shape)}",
        f"rank={tensor.ndim}",
        f"elements={tensor.numel()}",
        f"dtype={tensor.dtype}",
        f"device={tensor.device}",
    )


for tensor_name, value in {
    "scalar": scalar,
    "vector": vector,
    "matrix": matrix,
    "images": image_batch,
    "tokens": token_batch,
}.items():
    describe(tensor_name, value)
~~~

</details>

这些示例使用了两组常见符号：$B$ 表示批量大小，$C$ 表示通道数，$H \times W$ 表示空间尺寸，$L$ 表示序列长度，$D$ 表示特征宽度。这些字母并不是 Tensor 中保存的属性。PyTorch 只能看到各个轴的长度，程序员必须维护它们的含义。

**应用场景。** 一批 8 张 RGB 图像和一批 8 个 token 序列，在预处理后都可能成为三阶或四阶 Tensor，但各个轴支持的操作并不相同。将通道轴与空间轴交换，仍然可能得到一个语法上合法的 Tensor，却悄悄改变了模型的语义。

**对比总结。** Python 数字表示一个数值；Tensor 表示数值以及结构和执行元数据。阶数说明需要多少个索引，shape 说明索引的合法范围，轴语义则说明这些索引在问题中代表什么。

### **Tensor Shape、轴与布局** {#tensor-shape-axes-layout}

Tensor 的 **shape** 是由各轴长度组成的元组。**轴**是一条坐标方向，而轴的**语义角色**解释了沿该坐标变化意味着什么。考虑图像批量 $X \in \mathbb{R}^{B \times C \times H \times W}$：

$$
X[b,c,h,w]
$$

选取的是批量中第 $b$ 个样本、通道 $c$、第 $h$ 行和第 $w$ 列位置上的一个标量。Shape `[32, 3, 224, 224]` 只有与 `[batch, channel, height, width]` 的解释结合起来才真正有用。

![四阶 Tensor 可以为每个轴赋予独立含义，例如 batch、width、height 与 feature。](assets/tf-tensor-axis-order.png){fig-align="center" width="76%" fig-alt="一个四阶 Tensor 图，标注了 batch、width、height 和 feature 轴。"}

*图片来源：TensorFlow Core，[Introduction to Tensors](https://www.tensorflow.org/guide/tensor)，采用 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可。该 TensorFlow 图展示的是与框架无关的 Tensor 概念；本章代码使用 PyTorch。*

不同库和计算内核可能选择不同约定。PyTorch 视觉模型通常采用 channel-first 的 `[B, C, H, W]`，而一些系统使用 channel-last 的 `[B, H, W, C]`。序列模型通常使用 `[B, L, D]`，一些较早的循环网络接口则使用 `[L, B, D]`。可以通过轴置换在约定之间转换：

$$
[B,H,W,C] \xrightarrow{\operatorname{permute}(0,3,1,2)} [B,C,H,W].
$$

逻辑 shape 并不能完整描述物理存储。一个稠密 Tensor 具有底层的一维存储空间，并为每个轴保存一个 **stride**。如果 stride 元组是 $(s_1,\ldots,s_k)$，那么逻辑索引 $(i_1,\ldots,i_k)$ 对应的存储偏移量与下面的表达式成比例：

$$
\operatorname{offset}
= i_1s_1+i_2s_2+\cdots+i_ks_k.
$$

对于 shape 为 `[2, 3, 4]`、按行优先连续存储的 Tensor，stride 通常为 `[12, 4, 1]`：沿最后一个轴移动一步会前进一个存储元素，沿中间轴移动一步会前进四个元素，沿第一个轴移动一步会前进十二个元素。

`transpose` 与 `permute` 等操作通常不会复制数值，而是利用新的 shape 和 stride 元数据创建一个共享存储的 **view**。结果可能不连续。这种机制很高效，但后续如果某个运算假设连续存储，就可能需要 `.contiguous()`，或者由 `.reshape()` 隐式创建副本。

<details>
<summary><strong>PyTorch：检查 stride、view、轴置换与复制</strong></summary>

~~~python
import torch

x = torch.arange(24).reshape(2, 3, 4)
y = x.permute(0, 2, 1)  # [2, 4, 3], metadata changes; storage is shared.

print("x:", x.shape, x.stride(), x.is_contiguous())
print("y:", y.shape, y.stride(), y.is_contiguous())
print("shared storage:", x.untyped_storage().data_ptr() == y.untyped_storage().data_ptr())

# view() requires a compatible stride pattern. Flattening this permutation fails.
try:
    y.view(-1)
except RuntimeError as error:
    print("view failed:", str(error).split(".")[0])

# reshape() returns a view when possible and otherwise creates a contiguous copy.
y_flat = y.reshape(-1)
print("reshape is contiguous:", y_flat.is_contiguous())
print(
    "reshape shares storage:",
    y_flat.untyped_storage().data_ptr() == y.untyped_storage().data_ptr(),
)

# Make the copy explicit when a downstream kernel requires contiguous storage.
y_contiguous = y.contiguous()
assert y_contiguous.shape == (2, 4, 3)
assert y_contiguous.is_contiguous()
~~~

</details>

这种差异同时影响正确性和性能。`permute` 改变索引对应的轴；`reshape` 改变已有元素的分组方式；不应使用其中一个来模拟另一个。即使元素总数符合预期，错误 reshape 后的 Tensor 在语义上仍可能完全错误。

一种有效做法是在模块边界维护 **shape 不变量**：

~~~text
input images:       [B, C, H, W]
patch embeddings:   [B, L, D]
class logits:       [B, K]
targets:            [B]
~~~

使用 `assert logits.shape == (targets.shape[0], number_of_classes)` 之类的断言，可以把悄无声息的语义错误变成立即出现的失败。

**对比总结。** Shape 描述逻辑网格，stride 描述如何在存储空间中移动，轴名称描述其含义。`permute` 重排轴，`reshape` 重新组织元素，而 `.contiguous()` 会在必要时物化出兼容的内存顺序。

### **索引、重塑与广播** {#indexing-reshaping-broadcasting}

索引回答**保留哪些元素**；重塑回答**如何组织相同元素**；广播回答**具有兼容 shape 的 Tensor 如何参与逐元素运算**。三者相互关联，但不能彼此替代。

对于 shape 为 `[B, L, D]` 的 Tensor `x`：

| 表达式 | 输出 shape | 含义 |
|---|---|---|
| `x[0]` | `[L, D]` | 删除 batch 轴并选取一个样本 |
| `x[0:1]` | `[1, L, D]` | 保留长度为 1 的 batch 轴 |
| `x[:, -1]` | `[B, D]` | 选取每个样本的最后一个位置 |
| `x[..., 0]` | `[B, L]` | 选取最后一个轴上的第一个特征 |
| `x.unsqueeze(1)` | `[B, 1, L, D]` | 插入一个长度为 1 的轴 |
| `x.flatten(0, 1)` | `[B L, D]` | 合并相邻的 batch 与 sequence 轴 |

基础切片通常创建 view。布尔索引或整数数组高级索引可能分配新的 Tensor。对 view 进行原地修改也可能改变其底层 Tensor；这种行为在明确使用时很方便，在隐藏发生时则很危险。

重塑必须保持元素总数：

$$
\prod_j d_j = \prod_m d'_m.
$$

例如 `[B, L, H, d_h]` 可以变为 `[B, L, H d_h]`，但只有在最后两个轴确实分别表示注意力头和每个头的特征，并且存储顺序符合预期时，这个转换才有意义。

广播从**最后一个轴**开始对齐 shape。当两个对齐维度相等、其中一个为 1，或者其中一个维度不存在时，它们可以兼容。如果 $X$ 的 shape 为 `[B, L, D]`，$b$ 的 shape 为 `[D]`，那么：

$$
X + b:
[B,L,D] + [D]
\longrightarrow [B,L,D],
$$

因为 `[D]` 会表现得像 `[1, 1, D]`。扩展后的数值通常通过 stride 元数据表达，而不会真的复制到新的存储空间。

![广播将 shape 为 3 乘 1 的列 Tensor 与 shape 为 1 乘 4 的行 Tensor 组合，产生 3 乘 4 的结果。](assets/tf-broadcasting.png){fig-align="center" width="68%" fig-alt="一个广播图，将三元素列向量与四元素行向量相乘，形成三乘四矩阵。"}

*图片来源：TensorFlow Core，[Introduction to Tensors: Broadcasting](https://www.tensorflow.org/guide/tensor#broadcasting)，采用 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可。PyTorch 遵循相同的 NumPy 风格尾部轴规则，参见官方 [PyTorch broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)。*

广播的优势是能够消除显式循环，但它也可能生成合法却并非预期的输出。一个经典案例是把 `[B, 1]` 与 `[B]` 相加：按照尾部轴对齐，结果是 `[B, B]`，而不是 `[B, 1]`。

<details>
<summary><strong>PyTorch：按通道归一化并暴露一个静默 broadcasting 错误</strong></summary>

~~~python
import torch

# Four RGB images: [batch, channel, height, width].
images = torch.randn(4, 3, 16, 16)

# keepdim=True preserves [1, C, 1, 1], so every semantic axis remains visible.
channel_mean = images.mean(dim=(0, 2, 3), keepdim=True)
channel_std = images.std(dim=(0, 2, 3), keepdim=True).clamp_min(1e-6)
normalized = (images - channel_mean) / channel_std

assert channel_mean.shape == (1, 3, 1, 1)
assert normalized.shape == images.shape

# A legal but incorrect broadcast: [B, 1] + [B] -> [B, B].
scores = torch.randn(4, 1)
offsets = torch.arange(4, dtype=torch.float32)
wrong = scores + offsets
print("wrong shape:", wrong.shape)  # torch.Size([4, 4])

# Insert the intended singleton feature axis: [B] -> [B, 1].
correct = scores + offsets.unsqueeze(1)
assert correct.shape == scores.shape

# torch.broadcast_shapes can validate a planned elementwise operation explicitly.
assert torch.broadcast_shapes(images.shape, channel_mean.shape) == images.shape
~~~

</details>

对于 `sum`、`mean` 或 `max` 等归约操作，是否保留被归约的轴同样重要。`keepdim=True` 通常能让后续广播更安全，因为轴的语义位置仍然清晰可见。

**应用场景。** 广播可用于逐通道图像归一化、逐特征仿射变换、attention mask 和偏置相加。每一种用法都应该能够逐轴解释。如果输出意外增加了维度，应当先打印两个操作数的 shape，再检查具体数值。

**对比总结。** 索引选择数据，重塑重新组织索引系统，广播为逐元素运算虚拟扩展 singleton 维度。合法 shape 只能保证运算可以执行，不能保证它表达了预期语义。

### **矩阵乘法与爱因斯坦求和** {#matrix-multiplication-einstein-summation}

逐元素乘法和矩阵乘法回答的是不同问题。对于矩阵 $A,B \in \mathbb{R}^{M \times N}$，Hadamard 乘积保持 shape 不变：

$$
(A \odot B)_{ij}=A_{ij}B_{ij}.
$$

矩阵乘法则会收缩一个轴。如果 $A \in \mathbb{R}^{M \times K}$，$B \in \mathbb{R}^{K \times N}$，那么：

$$
C=AB \in \mathbb{R}^{M \times N},
\qquad
C_{ij}=\sum_{k=1}^{K}A_{ik}B_{kj}.
$$

共享的 $K$ 轴会消失，因为它的贡献被求和。这个**收缩轴**概念是线性层、点积注意力、卷积实现和许多损失函数的核心。

PyTorch 的 `torch.matmul` 与 `@` 对矩阵乘法进行了推广：

- 向量 `@` 向量返回标量点积；
- 矩阵 `@` 矩阵执行普通矩阵乘法；
- 对更高阶输入，把最后两个轴视为矩阵，并对前面的 batch 轴进行广播。

对于序列投影：

$$
X \in \mathbb{R}^{B \times L \times D_{in}},
\quad
W \in \mathbb{R}^{D_{in} \times D_{out}},
\quad
XW \in \mathbb{R}^{B \times L \times D_{out}}.
$$

每个 token 向量都会与同一个权重矩阵相乘。前面的 `[B, L]` 轴得到保留，$D_{in}$ 被收缩并替换为 $D_{out}$。

**爱因斯坦求和**可以显式表达这种轴逻辑。在下面的等式中：

$$
Y_{bld}=\sum_k X_{blk}W_{kd},
$$

只出现在输入而没有出现在输出中的索引会被求和；出现在输出中的索引会被保留。PyTorch 使用 `torch.einsum("blk,kd->bld", x, weight)` 表达相同运算。

<details>
<summary><strong>PyTorch：比较逐元素、矩阵、批量与 einsum 运算</strong></summary>

~~~python
import math
import torch

B, L, D_IN, D_OUT = 2, 5, 4, 6
x = torch.randn(B, L, D_IN)
weight = torch.randn(D_IN, D_OUT)

# The final input-feature axis is contracted with weight axis 0.
projected_matmul = x @ weight
projected_einsum = torch.einsum("blk,kd->bld", x, weight)

assert projected_matmul.shape == (B, L, D_OUT)
assert torch.allclose(projected_matmul, projected_einsum)

# Attention scores contract the per-head feature axis d.
HEADS, HEAD_DIM = 3, 8
queries = torch.randn(B, HEADS, L, HEAD_DIM)
keys = torch.randn(B, HEADS, L, HEAD_DIM)

scores = torch.einsum("bhid,bhjd->bhij", queries, keys) / math.sqrt(HEAD_DIM)
assert scores.shape == (B, HEADS, L, L)

# Elementwise multiplication does not contract an axis.
gated_queries = queries * torch.sigmoid(keys)
assert gated_queries.shape == queries.shape
~~~

</details>

当 `einsum` 能够澄清复杂收缩关系时最有价值，而不是仅仅用于缩短熟悉的代码。索引标签应遵循明确记录的约定，并通过断言检查重复维度。一个语法合法的等式仍然可能收缩错误的轴。

复杂度由收缩维度决定。执行 `[M, K] @ [K, N]` 大约需要 $MKN$ 组乘加运算，输出存储 $MN$ 个数值。前面的批量轴会成倍增加计算量，但不会改变收缩规则。

**应用场景。** 线性分类器把特征宽度收缩为类别 logits；注意力把 query 和 key 的特征收缩为位置两两之间的分数；双线性模型可以围绕可学习关系矩阵收缩两个特征轴。在编写代码前先写索引等式，通常能够立即暴露 shape 错误。

**对比总结。** 逐元素运算对齐并保留轴；矩阵乘法收缩一对轴；`einsum` 使用命名索引描述任意轴的保留与收缩。因此，shape 推理也是一种简洁的算法推理。

### **计算图** {#computation-graphs}

**计算图**把数值程序表示为一组依赖关系。Tensor 数值沿边流动，运算节点把输入转换为输出。如果标量目标是：

$$
q=3a^3-b^2,
$$

那么前向计算包含幂、乘法与减法运算。计算图记录足够的局部信息，从而回答改变 $a$ 或 $b$ 会如何改变 $q$。

一次模型调用的前馈执行通常形成一个**有向无环图（DAG）**：边从较早数值指向较晚数值，在前向时间上不存在输出通过循环依赖自身。循环神经网络在有限序列上重复状态转换后展开，仍然会形成 DAG。

![PyTorch 会为创建输出 Tensor 的运算记录对应的反向函数节点。](assets/pytorch-autograd-dag.png){fig-align="center" width="48%" fig-alt="一个 PyTorch autograd 图，其中包含幂、乘法与减法的反向节点。"}

*图片来源：PyTorch Tutorials，[A Gentle Introduction to `torch.autograd`](https://docs.pytorch.org/tutorials/beginner/blitz/autograd_tutorial.html#computational-graph)。*

在 PyTorch 中，eager 运算会立即执行，同时 autograd 动态记录可微依赖。在一次前向传播中，它会：

1. 计算每个请求的 Tensor 结果；
2. 当需要追踪梯度时，为非叶子输出附加 `grad_fn`；
3. 保存反向公式所需的部分数值或元数据。

这个图是**动态的**：Python 控制流可以在不同调用中选择不同运算。一次普通反向传播结束后，保存的计算图状态会被释放；下一次前向传播会重新建立新图。这与静态源代码图不同：autograd 图描述的是针对具体 Tensor 数值真正执行过的运算。

PyTorch 会区分：

- **叶子 Tensor**：不是由被追踪运算产生的 Tensor，通常包括可训练参数；
- **非叶子 Tensor**：具有 `grad_fn` 的中间结果；
- **根节点**：反向遍历开始的位置，通常是一个标量损失。

<details>
<summary><strong>PyTorch：检查叶子 Tensor 与动态计算图</strong></summary>

~~~python
import torch

a = torch.tensor([2.0, 3.0], requires_grad=True)
b = torch.tensor([6.0, 4.0], requires_grad=True)

q = 3 * a**3 - b**2
loss = q.sum()

print("a is leaf:", a.is_leaf, "grad_fn:", a.grad_fn)
print("q is leaf:", q.is_leaf, "grad_fn:", type(q.grad_fn).__name__)
print("loss grad_fn:", type(loss.grad_fn).__name__)

loss.backward()

# dq/da = 9a^2 and dq/db = -2b.
assert torch.allclose(a.grad, 9 * a.detach() ** 2)
assert torch.allclose(b.grad, -2 * b.detach())
print("gradient for a:", a.grad)
print("gradient for b:", b.grad)
~~~

</details>

计算图与 `nn.Module` 树并不是同一个概念。Module 树描述注册的组件与参数；计算图描述某一次执行过程，并且可以多次调用同一模块、跳过某个分支，或者让一个 Tensor 沿多条路径被复用。

计算图记录机制也解释了为什么某些原地运算不安全。反向传播可能需要较早的 Tensor 数值。如果该数值被覆盖，PyTorch 会抛出版本检查错误；在隔离不充分的自定义代码中，预期导数甚至可能无法重建。

**对比总结。** 源代码描述可能的执行过程，Module 树描述拥有的组件，计算图描述真正发生的数值依赖。计算图把前向数值与稍后需要的局部导数规则连接起来。

### **PyTorch 中的自动微分** {#automatic-differentiation-pytorch}

**自动微分（AD）**通过把精确的导数规则应用于已执行的基本运算来计算导数。它不是操作代数表达式的符号微分，也不是通过扰动输入估计斜率的有限差分。

深度学习通常包含大量参数和一个标量损失。**反向模式自动微分**很适合这种多输入、单输出结构。假设中间状态满足：

$$
\mathbf{h}^{(0)}=\mathbf{x},
\qquad
\mathbf{h}^{(\ell)}=f^{(\ell)}(\mathbf{h}^{(\ell-1)}),
\qquad
\mathcal{L}=g(\mathbf{h}^{(L)}),
$$

反向模式利用向量-Jacobian 乘积把上游敏感度向后传播：

$$
\mathbf{v}_{\ell-1}
=
\left(\frac{\partial \mathbf{h}^{(\ell)}}
{\partial \mathbf{h}^{(\ell-1)}}\right)^{\!T}
\mathbf{v}_{\ell}.
$$

系统通常不会真的物化完整 Jacobian。每个运算接收一个上游向量，并返回其输入需要的梯度贡献。第 04 章会详细推导这一机制与反向传播；本节重点说明 PyTorch 的接口契约。

在叶子 Tensor 上设置 `requires_grad=True`，表示 autograd 应追踪能够影响它的运算。对标量输出调用 `.backward()` 会开始反向遍历。梯度会累加到符合条件的叶子 Tensor 的 `.grad` 中，因此训练代码必须使用 `optimizer.zero_grad()` 清理梯度，或者把它们设为 `None`。

几个重要接口具有不同用途：

| 接口 | 效果 | 典型用途 |
|---|---|---|
| `tensor.detach()` | 返回与当前计算图断开的 Tensor | 阻断梯度路径或记录数值 |
| `torch.no_grad()` | 在代码块中禁止记录计算图 | 参数更新或评估代码 |
| `torch.inference_mode()` | 更强的仅推理优化模式 | 部署或评估前向传播 |
| `torch.autograd.grad()` | 返回指定梯度，而不仅依赖 `.grad` | 惩罚项、分析或高阶方法 |
| `retain_graph=True` | 在反向传播后保留图状态 | 少数需要重复遍历的情况，但会增加内存 |

广播也会影响反向规则。如果偏置 $b \in \mathbb{R}^{D}$ 被广播到批量 $X \in \mathbb{R}^{B \times D}$ 上，那么 $b$ 的梯度会对被扩展的 batch 轴求和：

$$
\frac{\partial \mathcal{L}}{\partial b_d}
=
\sum_{i=1}^{B}
\frac{\partial \mathcal{L}}{\partial Y_{id}}.
$$

<details>
<summary><strong>PyTorch：使用有限差分验证 autograd 梯度</strong></summary>

~~~python
import torch

# float64 makes the numerical finite-difference comparison more precise.
x = torch.tensor([1.5, -2.0], dtype=torch.float64)
target = torch.tensor(0.4, dtype=torch.float64)
w = torch.tensor([0.3, -0.7], dtype=torch.float64, requires_grad=True)


def objective(weights: torch.Tensor) -> torch.Tensor:
    prediction = (x * weights).sum()
    return (prediction - target) ** 2


loss = objective(w)
loss.backward()
autograd_gradient = w.grad.detach().clone()

# Central finite differences are a diagnostic, not the training algorithm.
epsilon = 1e-6
finite_difference = torch.empty_like(w)
with torch.no_grad():
    for index in range(w.numel()):
        plus = w.detach().clone()
        minus = w.detach().clone()
        plus[index] += epsilon
        minus[index] -= epsilon
        finite_difference[index] = (
            objective(plus) - objective(minus)
        ) / (2 * epsilon)

print("autograd:", autograd_gradient)
print("finite difference:", finite_difference)
print("maximum error:", (autograd_gradient - finite_difference).abs().max().item())
assert torch.allclose(autograd_gradient, finite_difference, atol=1e-8, rtol=1e-6)
~~~

</details>

有限差分适合检查小型自定义运算，但它扩展性差，并且会受到截断误差和浮点误差影响。自动微分复用真正执行的程序，在基本运算可微且数值行为合理的前提下，以工作精度计算导数。

常见 autograd 错误包括：过早 detach 数值、在数值参与损失前使用 `.item()` 转为 Python 数字、原地修改被保存的 Tensor、忘记梯度会累加，或者期望非叶子 Tensor 的 `.grad` 自动得到保留。

**对比总结。** 有限差分通过重复评估估计导数；符号微分变换公式；自动微分沿已执行计算图组合局部导数规则。PyTorch 默认的反向模式非常适合标量损失训练。

### **参数、Module 与模型组合** {#parameters-modules-model-composition}

`nn.Module` 不仅是一个 Python 函数。它是一个所有权边界，用于注册可训练参数、持久 buffer 和嵌套模块。注册机制让 PyTorch 能够发现优化、设备迁移、序列化和模式切换所需要的系统状态。

当一个 `nn.Parameter` 被赋值为模块属性时，它就会成为模块参数集合的一部分。**Buffer** 是应当随模块迁移和序列化、但默认不参与优化的持久 Tensor 状态，例如归一化统计量、mask 或固定位置数值。

调用 `model(inputs)` 会先执行模块的 `__call__` 机制，再调用 `forward`；直接调用 `model.forward(inputs)` 会绕过 hook，通常不应这样使用。嵌套模块会形成树结构，而一次沿该树执行的 forward 调用会建立前面讨论的动态计算图。

`state_dict` 把注册状态名称映射到 Tensor。它通常包含参数与持久 buffer，却不包含任意 Python 属性。因此，可以直接观察注册行为：

<details>
<summary><strong>PyTorch：注册参数、buffer 与嵌套模块</strong></summary>

~~~python
import torch
from torch import nn


class StandardizedRegressor(nn.Module):
    def __init__(self, feature_mean: torch.Tensor, feature_std: torch.Tensor):
        super().__init__()

        # Buffers move with .to(device) and appear in state_dict,
        # but an optimizer will not update them.
        self.register_buffer("feature_mean", feature_mean)
        self.register_buffer("feature_std", feature_std.clamp_min(1e-6))

        # Assigning nn.Modules registers their Parameters recursively.
        self.predictor = nn.Sequential(
            nn.Linear(feature_mean.numel(), 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        assert features.shape[-1] == self.feature_mean.numel()
        standardized = (features - self.feature_mean) / self.feature_std
        return self.predictor(standardized).squeeze(-1)


model = StandardizedRegressor(
    feature_mean=torch.tensor([10.0, 2.0, -1.0]),
    feature_std=torch.tensor([2.0, 0.5, 4.0]),
)
batch = torch.tensor([[12.0, 2.5, 3.0], [8.0, 1.5, -5.0]])
predictions = model(batch)

print("output shape:", predictions.shape)
print("parameter names:", [name for name, _ in model.named_parameters()])
print("buffer names:", [name for name, _ in model.named_buffers()])
print("state keys:", list(model.state_dict()))

assert predictions.shape == (2,)
assert "feature_mean" in model.state_dict()
assert "predictor.0.weight" in model.state_dict()
~~~

</details>

`model.train()` 和 `model.eval()` 会改变 dropout 与 batch normalization 等模块的行为；它们不会打开或关闭梯度记录。评估通常同时需要 `model.eval()` 与 `torch.inference_mode()`，因为二者控制的是不同机制。

模块组合应维护清晰的接口契约。一个模块应说明预期 shape、dtype、数值范围与输出含义。小模块更容易测试，但过度拆分也会隐藏主要数据流。合适的边界通常应当拥有一个内聚的转换过程及其状态。

**对比总结。** Tensor 保存数值状态，`Parameter` 标记可优化状态，buffer 标记持久但不优化的 Tensor 状态，`Module` 则拥有并组合这些状态。Module 树是持久结构，计算图是每次执行产生的行为。

### **Dataset、DataLoader 与 Mini-Batch** {#datasets-dataloaders-mini-batches}

模型消费的是 Tensor，但真实数据最初可能表现为文件、记录、序列、标签和转换规则。PyTorch 把输入系统拆分为不同职责：

- **Dataset** 定义如何识别和获取一个样本；
- **Sampler** 定义请求哪些样本索引以及请求顺序；
- **collate function** 把取出的样本组合成 batch；
- **DataLoader** 协调分批、迭代、worker 与内存传输选项。

Map-style `Dataset` 通常实现 `__len__` 与 `__getitem__`。`IterableDataset` 则逐个产生数据流，更适合无法或不希望随机索引的场景，例如日志、远程数据流或动态生成的样本。这种选择会改变 shuffle 和多 worker 分区的实现方式。

Mini-batch 同时具有统计与系统意义。对于逐样本损失 $\ell_i(\theta)$，大小为 $B$ 的 batch 使用下面的表达式估计完整数据梯度：

$$
\widehat{\mathbf{g}}_B
=
\frac{1}{B}\sum_{i \in \mathcal{B}}
\nabla_{\theta}\ell_i(\theta).
$$

更大的 batch 通常会降低采样噪声并提升硬件利用率，直到内存或通信成为瓶颈。它不会自动改善泛化，而且会改变每个 epoch 的优化器更新次数。因此，需要同时解释 batch size、学习率、归一化行为与 scheduler 的计数单位。

Collate 阶段定义 batch 契约。固定尺寸图像可以直接堆叠为 `[B, C, H, W]`。变长序列如果没有 padding、packing、truncation 或 nested/ragged representation 等策略，就无法直接堆叠。Padding 还需要 mask，让后续运算能够区分真实位置和占位符。

<details>
<summary><strong>PyTorch：构建变长 Dataset 与支持 mask 的 collate function</strong></summary>

~~~python
import torch
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset


class SequenceDataset(Dataset):
    def __init__(self):
        self.examples = [
            (torch.tensor([4, 8, 2, 9]), 1),
            (torch.tensor([3, 5]), 0),
            (torch.tensor([7, 1, 6]), 1),
            (torch.tensor([2, 2, 4, 8, 5]), 0),
        ]

    def __len__(self) -> int:
        return len(self.examples)

    def __getitem__(self, index: int):
        tokens, label = self.examples[index]
        return tokens, torch.tensor(label, dtype=torch.long)


def pad_collate(batch):
    """Convert a list of variable-length examples into dense tensors."""
    token_sequences, labels = zip(*batch)
    lengths = torch.tensor([sequence.numel() for sequence in token_sequences])
    padded = pad_sequence(token_sequences, batch_first=True, padding_value=0)

    # True marks a real token; False marks padding.
    positions = torch.arange(padded.shape[1]).unsqueeze(0)
    attention_mask = positions < lengths.unsqueeze(1)
    return {
        "tokens": padded,                  # [B, L_max]
        "attention_mask": attention_mask,  # [B, L_max]
        "lengths": lengths,                # [B]
        "labels": torch.stack(labels),     # [B]
    }


generator = torch.Generator().manual_seed(7)
loader = DataLoader(
    SequenceDataset(),
    batch_size=3,
    shuffle=True,
    collate_fn=pad_collate,
    num_workers=0,
    generator=generator,
)

batch = next(iter(loader))
for name, value in batch.items():
    print(name, tuple(value.shape), value.dtype)

assert batch["tokens"].shape == batch["attention_mask"].shape
assert torch.equal(batch["attention_mask"].sum(dim=1), batch["lengths"])
~~~

</details>

对于彼此独立的训练样本，通常应启用 shuffle；为了确定性评估，则应关闭 shuffle。`drop_last=True` 会丢弃较小的最后一个 batch，这可能适合对 shape 敏感的训练，却会改变实际参与训练的样本数。多个 worker 可以让数据加载与计算重叠，但每个 worker 必须得到合理的随机种子，并且不能重复同一段 iterable 数据流。

锁页主机内存与异步设备复制能够改善部分 GPU pipeline，但前提是完整传输路径都支持这些机制。更多 worker 并不总是更快：对于小型内存数据集，进程协调成本可能高于数据准备成本。官方 [Datasets & DataLoaders tutorial](https://docs.pytorch.org/tutorials/beginner/basics/data_tutorial.html) 给出了核心 API 行为。

**应用场景。** 数据边界是 tokenization、augmentation、padding、标签转换和样本元数据变为 Tensor 的位置。独立测试 Dataset 与 collate function，可以避免模型代码掩盖格式错误的样本或不一致的 shape。

**对比总结。** Dataset 定义样本，Sampler 定义顺序，collate 定义 batch 结构，DataLoader 组织迭代。Mini-batch 既是梯度估计器，也是硬件工作单元。

### **设备、数据类型与数值精度** {#devices-data-types-numerical-precision}

只有操作数满足计算内核的 device 与 dtype 要求，Tensor 运算才能执行。PyTorch 不会在算术运算时悄悄把 CPU Tensor 移到 GPU，因为隐式传输代价高昂而且难以推理。模型状态与输入 batch 必须显式迁移。

常见设备选择包括：

- `cpu`：适合可移植性、预处理、小模型和包含大量控制流的工作；
- `cuda`：用于 NVIDIA 加速器；
- `mps`：用于受支持的 Apple Silicon 加速；
- 其他后端，例如可用时的 XPU 或专用加速器。

设备放置不仅涉及计算速度。一个训练步骤可能受到 host-to-device 传输、加速器内存、kernel launch 开销或同步的限制。在紧密循环中调用 `.item()`、打印 GPU Tensor，或者把数据移回 CPU，都可能强制同步并降低吞吐量。

**Dtype** 控制数值范围、精度、存储大小和可用计算内核：

| Dtype | 每元素字节数 | 典型用途 | 主要注意点 |
|---|---:|---|---|
| `float64` | 8 | 科学计算检查、高精度诊断 | 在许多加速器上更慢且更占空间 |
| `float32` | 4 | 标准训练与稳定累加 | 比低精度占用更多内存 |
| `bfloat16` | 2 | 具有近似 float32 指数范围的混合精度训练 | 尾数位更少 |
| `float16` | 2 | 加速混合精度与推理 | 指数范围更窄，存在上溢/下溢风险 |
| `int64` | 8 | 许多 API 需要的类别标签与索引 | 不可微 |
| `bool` | 1 | Mask 与逻辑选择 | 算术含义必须明确 |

一个稠密 Tensor 的数据载荷内存大约为：

$$
\text{bytes}
=
\operatorname{numel}(X)
\times
\operatorname{element\_size}(X).
$$

这个估计不包含 allocator 开销，也不包含梯度、优化器状态、保存的激活值和临时 workspace。一个配合 Adam 使用的模型参数，通常会在参数本身之外产生多个额外 Tensor。

低精度可以加速许多工作负载并节省内存，但不是所有运算都应使用同一个 dtype。混合精度系统通常会使用 autocast、较高精度累加，并在必要时进行 gradient scaling。第 19 章会详细讨论该训练系统。

数值稳定性不仅取决于 dtype，也取决于算法形式。朴素 softmax：

$$
p_i=\frac{e^{z_i}}{\sum_j e^{z_j}}
$$

在 logits 很大时可能上溢。减去最大值在数学上等价，而且数值更加稳定：

$$
p_i
=
\frac{e^{z_i-m}}{\sum_j e^{z_j-m}},
\qquad
m=\max_j z_j.
$$

生产级 `torch.softmax` 使用稳定实现；手动展开熟悉公式可能失去这种保护。

<details>
<summary><strong>PyTorch：选择设备、估算 Tensor 内存并检查数值稳定性</strong></summary>

~~~python
import torch
from torch import nn


def available_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = available_device()
model = nn.Linear(768, 10).to(device)
features = torch.randn(32, 128, 768, device=device)
logits = model(features)
assert logits.shape == (32, 128, 10)

payload_mebibytes = features.numel() * features.element_size() / 2**20
print("device:", device)
print("activation payload MiB:", round(payload_mebibytes, 2))

# A direct exponential overflows even though the probability is well defined.
large_logits = torch.tensor([1000.0, 1001.0])
naive = torch.exp(large_logits) / torch.exp(large_logits).sum()
stable = torch.softmax(large_logits, dim=0)

print("naive softmax:", naive)
print("stable softmax:", stable)
assert torch.isnan(naive).any()
assert torch.isfinite(stable).all()
~~~

</details>

代码会把模型和特征移动到同一个可用设备。Target 还必须符合损失函数契约：对于 `CrossEntropyLoss`，类别索引 target 通常为 `int64`，模型 logits 则是浮点数。把所有 Tensor 都转换为同一个 dtype 并不是正确的设备策略。

**对比总结。** Device 决定在哪里执行计算，dtype 决定表示范围、精度与内存，算法形式决定如何遇到舍入误差和指数范围限制。性能与稳定性需要同时设计这三个方面。

### **随机性、Checkpoint 与可复现性** {#randomness-checkpoints-reproducibility}

深度学习实验包含多种状态。模型参数只是其中一部分：

- Python、NumPy 与 PyTorch 随机数生成器状态；
- 数据划分标识、样本顺序与数据增强随机性；
- 优化器的动量和方差估计；
- Scheduler、gradient scaler 和 early stopping 状态；
- 训练 step、epoch、配置、代码版本与运行环境；
- 模型参数与持久 buffer。

设置随机种子可以让运行过程更可控，但不能保证在不同 PyTorch 版本、平台、设备或 kernel 上实现逐 bit 相同。一些加速器算法本身具有非确定性，而确定性替代方案可能更慢。因此，可复现性声明必须说明环境和所追求的等价层级：精确重放、统计一致的结果，或者科学结论的再现。官方 [PyTorch reproducibility notes](https://docs.pytorch.org/docs/stable/notes/randomness.html) 明确说明了这一限制。

**Checkpoint** 是继续运行或检查实验所需状态的序列化结果。如果架构与预处理方式已知，只保存 `model.state_dict()` 就足以用于推理。想要精确恢复训练，通常还需要优化器状态、进度计数器、随机状态和数据顺序状态。

<details>
<summary><strong>PyTorch：创建并恢复完整训练 checkpoint</strong></summary>

~~~python
import io
import random

import numpy as np
import torch
from torch import nn

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 1))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-2)
features = torch.randn(6, 3)
targets = torch.randn(6, 1)

# Perform one update so the optimizer also contains nontrivial state.
loss = nn.functional.mse_loss(model(features), targets)
optimizer.zero_grad()
loss.backward()
optimizer.step()
reference_prediction = model(features).detach().clone()

data_generator = torch.Generator().manual_seed(99)
checkpoint = {
    "model_state": model.state_dict(),
    "optimizer_state": optimizer.state_dict(),
    "epoch": 1,
    "config": {"input_width": 3, "hidden_width": 5, "learning_rate": 1e-2},
    "python_rng_state": random.getstate(),
    "numpy_rng_state": np.random.get_state(),
    "torch_rng_state": torch.get_rng_state(),
    "data_generator_state": data_generator.get_state(),
}

# BytesIO keeps this example self-contained; a real run writes an atomic file.
buffer = io.BytesIO()
torch.save(checkpoint, buffer)
buffer.seek(0)

# weights_only=False is appropriate only for a trusted full-state checkpoint.
loaded = torch.load(buffer, map_location="cpu", weights_only=False)
restored_model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 1))
restored_optimizer = torch.optim.AdamW(restored_model.parameters(), lr=1e-2)
restored_model.load_state_dict(loaded["model_state"])
restored_optimizer.load_state_dict(loaded["optimizer_state"])

random.setstate(loaded["python_rng_state"])
np.random.set_state(loaded["numpy_rng_state"])
torch.set_rng_state(loaded["torch_rng_state"])
data_generator.set_state(loaded["data_generator_state"])

restored_prediction = restored_model(features).detach()
assert torch.allclose(reference_prediction, restored_prediction)
print("restored epoch:", loaded["epoch"])
print("predictions restored:", True)
~~~

</details>

完整 checkpoint 使用 Python 序列化，只能从可信来源加载。分发模型权重时，应优先使用 state dictionary 和当前可用的最安全加载模式。Checkpoint 也不会自动保存外部依赖：数据文件、词表版本、预处理代码和架构定义必须单独进行版本控制。

在工程上，checkpoint 写入应当是原子的：先写临时文件并 flush，只有成功后才进行重命名。最新的可恢复 checkpoint 应与最佳验证 checkpoint 分开保存，因为二者解决不同需求。损坏或只写入一部分的“最佳”文件不应成为唯一恢复路径。

**应用场景。** 可复现性不仅服务于论文，也服务于调试。当损失在第 18,400 个 step 突然升高时，完整 checkpoint 与配置能够帮助区分数据相关故障、偶发硬件问题和数值问题。

**对比总结。** Seed 控制初始随机流，确定性设置限制算法选择，checkpoint 捕获执行状态，实验记录解释运行环境与决策。它们无法相互替代。

### **本章对比与总结** {#chapter-comparison-summary}

当数值始终与接口契约结合时，Tensor 编程才会可靠。执行运算前，应识别每个轴；执行后，应预测输出 shape；进行训练时，应验证计算图连通性；进行系统工作时，应考虑 dtype、device、数据顺序与持久状态。

| 概念 | 首要问题 | 常见错误 |
|---|---|---|
| 阶数与 shape | 有多少个轴，每个轴有多长？ | 混淆 Tensor 阶数与矩阵秩 |
| 轴语义 | 沿每个轴移动会改变什么？ | 轴置换合法但语义错误 |
| Stride 与 layout | 逻辑索引如何映射到存储？ | 假设所有 view 都连续 |
| 索引与 reshape | 保留哪些数值，它们如何分组？ | 删除或合并错误的语义轴 |
| Broadcasting | 哪些 singleton 轴被虚拟扩展？ | 产生合法却并非预期的更大 Tensor |
| 矩阵收缩 | 哪些轴得到保留，哪些轴被求和？ | 错误收缩 feature、token 或 head 轴 |
| 计算图 | 哪些实际执行的运算把损失连接到参数？ | Detach 或覆盖反向传播需要的数值 |
| Autograd | 反向模式敏感度应在哪里累加？ | 使用旧的累积梯度或没有启用追踪 |
| Module 状态 | 哪些状态需要共同优化、迁移和序列化？ | 未注册 Tensor 或直接调用 `forward` |
| 数据 pipeline | 样本如何变成合法 batch？ | Padding 没有 mask，或 worker 重复数据流 |
| Device 与 dtype | Kernel 在哪里、以何种表示运行？ | 设备不匹配、上溢或不恰当类型转换 |
| 可复现性 | 解释或恢复运行需要哪些状态？ | 只保存权重，丢失优化器与数据顺序 |

本章的主要结论是：

1. Tensor 是数值数据以及 shape、dtype、device、layout 和可选 autograd 元数据的组合。
2. Shape 兼容只是必要条件；轴语义决定运算是否有意义。
3. View 可以利用不同 stride 共享存储，而 reshape 有时必须创建副本。
4. Broadcasting 从尾部维度开始对齐，并可能悄悄扩展输出 shape。
5. 理解矩阵乘法与 `einsum` 的最佳方式，是区分哪些轴被保留、哪些轴被收缩。
6. PyTorch 根据真正执行的运算建立动态计算图。
7. 反向模式 autograd 组合局部向量-Jacobian 乘积，并在被追踪的叶子 Tensor 中累加梯度。
8. Module 注册参数、buffer 与子模块，使状态能够得到一致的优化、迁移与序列化。
9. Dataset、采样、collation 与设备传输共同构成模型行为契约的一部分。
10. Seed、确定性设置、checkpoint 与环境记录分别解决可复现性的不同部分。

下一章会利用这些契约构造神经网络基本组件：线性转换、激活函数、embedding、残差路径、门控、归一化与可复用模块。
